# Goodbooks: collaborative filtering and content recommendation

Dataset: https://github.com/zygmuntz/goodbooks-10k

The setup cell accepts a project ZIP or a public GitHub repository URL. The ZIP contains requirements.txt, config.py, run_experiments.py, and the algorithms, preprocessing and evaluation folders.


In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

REPOSITORY_URL = ""  # Optional: paste your public project GitHub URL here.
if Path("/content").exists():
    work = Path("/content/goodbooks_project")
    if REPOSITORY_URL:
        if not work.exists():
            subprocess.run(["git", "clone", REPOSITORY_URL, str(work)], check=True)
    elif not work.exists():
        from google.colab import files
        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload exactly one project ZIP.")
        work.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            root = work.resolve()
            for entry in archive.infolist():
                if not (root / entry.filename).resolve().is_relative_to(root):
                    raise ValueError("Unsafe ZIP path")
            archive.extractall(work)
    candidates = list(work.rglob("run_experiments.py"))
    if not candidates:
        raise FileNotFoundError("The ZIP must contain the project source.")
    project = candidates[0].parent
else:
    project = Path.cwd()
    if not (project / "run_experiments.py").exists():
        project = project.parent
print("Project directory:", project)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(project / "requirements.txt")], check=True)


## Validate and run

Defaults: 500 users and 2,000 books. TUNE=True selects neighbourhood sizes on validation NDCG. Tags are disabled because their annotation timestamps are unavailable.


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=project, check=True)


In [ ]:
MAX_USERS = 500
MAX_ITEMS = 2000
EVAL_USERS = 100
EPOCHS = 20
SEED = 42
TUNE = True
command = [sys.executable, "run_experiments.py", "--max-users", str(MAX_USERS),
           "--max-items", str(MAX_ITEMS), "--eval-users", str(EVAL_USERS),
           "--epochs", str(EPOCHS), "--seed", str(SEED)]
if TUNE:
    command.append("--tune")
subprocess.run(command, cwd=project, check=True)


In [ ]:
import pandas as pd
from IPython.display import display, Image
results = pd.read_csv(project / "results/metrics.csv")
display(results.round(4))
display(Image(filename=str(project / "results/comparison.png")))


In [ ]:
import json
recommendations = json.loads((project / "results/recommendations.json").read_text())
for model, examples in recommendations.items():
    print("\n", model)
    if examples:
        print("User:", examples[0]["user"])
        for rank, title in enumerate(examples[0]["books"], 1):
            print(f"{rank:2}. {title}")


## Evaluation notes

Results compare ranking quality, coverage, diversity and running time. RMSE applies only to rating predictors; content probabilities are not star ratings. Unobserved books have unknown relevance. The full protocol and limitations are documented in README.md.


In [ ]:
import shutil
archive = shutil.make_archive(str(project / "experiment_results"), "zip", project / "results")
print("Results archive:", archive)
if Path("/content").exists():
    from google.colab import files
    files.download(archive)
